In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

SILVER_TABLE = "lh_silver.dbo.silver_flights"


def read_bronze(table: str):
    for name in (f"lh_bronze.dbo.{table}", f"lh_bronze.{table}"):
        try:
            return spark.table(name)
        except Exception:
            continue
    raise ValueError(f"Could not read {table} from lh_bronze")


def table_exists(name: str) -> bool:
    try:
        spark.table(name).limit(0).collect()
        return True
    except Exception:
        return False


def num(c):
    return F.col(c).cast("double")


def to_int(c):
    return num(c).cast("int")


def clean_str(c):
    trimmed = F.trim(F.col(c))
    return F.when(trimmed == "", F.lit(None)).otherwise(trimmed)


def hhmm_to_minutes(c):
    v = to_int(c)
    return (F.floor(v / 100) * 60 + (v % 100)).cast("int")


bronze = read_bronze("bronze_ontime")

required = [
    "_period_key", "_source_file", "_ingested_at",
    "year", "quarter", "month", "dayof_month", "day_of_week", "flight_date",
    "reporting_airline", "dot_id_reporting_airline", "flight_number_reporting_airline", "tail_number",
    "origin_airport_id", "origin", "origin_city_name", "origin_state",
    "dest_airport_id", "dest", "dest_city_name", "dest_state",
    "crs_dep_time", "dep_delay", "dep_delay_minutes", "dep_del15", "taxi_out",
    "crs_arr_time", "taxi_in", "arr_delay", "arr_delay_minutes", "arr_del15",
    "cancelled", "cancellation_code", "diverted",
    "crs_elapsed_time", "actual_elapsed_time", "air_time", "distance",
    "carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay",
]
missing = [c for c in required if c not in bronze.columns]
if missing:
    raise ValueError(f"Missing Bronze columns: {missing}")

bronze_periods = bronze.groupBy("_period_key").agg(F.max("_ingested_at").alias("bronze_ingested_at"))
silver_exists = table_exists(SILVER_TABLE)

if silver_exists:
    silver_periods = (
        spark.table(SILVER_TABLE)
        .groupBy("period_key")
        .agg(F.max("_bronze_ingested_at").alias("silver_ingested_at"))
    )
    periods_to_process = [
        r["_period_key"]
        for r in (
            bronze_periods
            .join(silver_periods, bronze_periods["_period_key"] == silver_periods["period_key"], "left")
            .filter(F.col("silver_ingested_at").isNull() | (F.col("bronze_ingested_at") > F.col("silver_ingested_at")))
            .select("_period_key")
            .collect()
        )
    ]
else:
    periods_to_process = [r["_period_key"] for r in bronze_periods.select("_period_key").collect()]

periods_to_process = sorted(periods_to_process)
print(f"Months to process: {len(periods_to_process)}")

if not periods_to_process:
    print("Silver is already up to date. Nothing to do.")
else:
    print(f"Range: {periods_to_process[0]} to {periods_to_process[-1]}")

    src = bronze.filter(F.col("_period_key").isin(periods_to_process))

    is_cancelled = num("cancelled") == 1
    is_diverted = num("diverted") == 1
    is_operated = (~is_cancelled) & (~is_diverted)

    sched_dep_min = hhmm_to_minutes("crs_dep_time")
    sched_arr_min = hhmm_to_minutes("crs_arr_time")
    day_start_seconds = F.unix_timestamp(F.col("flight_date"), "yyyy-MM-dd")
    sched_dep_seconds = day_start_seconds + sched_dep_min * 60

    cancel_code = F.when(is_cancelled, clean_str("cancellation_code"))
    arr_delay = to_int("arr_delay")
    dep_delay = to_int("dep_delay")
    missing_arrival_data = is_operated & (F.col("arr_delay").isNull() | (F.trim(F.col("arr_delay")) == "") | F.col("air_time").isNull() | (F.trim(F.col("air_time")) == ""))

    silver = src.select(
        F.col("_period_key").alias("period_key"),
        F.to_date("flight_date", "yyyy-MM-dd").alias("flight_date"),
        to_int("year").alias("year"),
        to_int("quarter").alias("quarter"),
        to_int("month").alias("month"),
        to_int("dayof_month").alias("day_of_month"),
        to_int("day_of_week").alias("day_of_week"),

        clean_str("reporting_airline").alias("airline_code"),
        to_int("dot_id_reporting_airline").alias("airline_dot_id"),
        clean_str("flight_number_reporting_airline").alias("flight_number"),
        clean_str("tail_number").alias("tail_number"),

        to_int("origin_airport_id").alias("origin_airport_id"),
        clean_str("origin").alias("origin_airport_code"),
        clean_str("origin_city_name").alias("origin_city_name"),
        clean_str("origin_state").alias("origin_state_code"),
        to_int("dest_airport_id").alias("dest_airport_id"),
        clean_str("dest").alias("dest_airport_code"),
        clean_str("dest_city_name").alias("dest_city_name"),
        clean_str("dest_state").alias("dest_state_code"),

        F.when(is_operated, F.lit("Operated"))
         .when(is_cancelled, F.lit("Cancelled"))
         .otherwise(F.lit("Diverted")).alias("flight_status"),
        is_cancelled.alias("is_cancelled"),
        is_diverted.alias("is_diverted"),
        cancel_code.alias("cancellation_code"),
        F.when(cancel_code == "A", F.lit("Carrier"))
         .when(cancel_code == "B", F.lit("Weather"))
         .when(cancel_code == "C", F.lit("National Air System"))
         .when(cancel_code == "D", F.lit("Security")).alias("cancellation_reason"),

        F.timestamp_seconds(sched_dep_seconds).alias("scheduled_departure_ts"),
        F.when(~is_cancelled & dep_delay.isNotNull(), F.timestamp_seconds(sched_dep_seconds + dep_delay * 60)).alias("actual_departure_ts"),
        (F.floor(sched_dep_min / 60) % 24).cast("int").alias("scheduled_departure_hour"),
        (F.floor(sched_arr_min / 60) % 24).cast("int").alias("scheduled_arrival_hour"),

        F.when(~is_cancelled, dep_delay).alias("departure_delay_min"),
        F.when(~is_cancelled, to_int("dep_delay_minutes")).alias("departure_delay_nonneg_min"),
        F.when(~is_cancelled, num("dep_del15") == 1).alias("is_departure_delayed_15"),
        F.when(is_operated, arr_delay).alias("arrival_delay_min"),
        F.when(is_operated, to_int("arr_delay_minutes")).alias("arrival_delay_nonneg_min"),
        F.when(is_operated & ~missing_arrival_data, num("arr_del15") == 1).alias("is_arrival_delayed_15"),
        F.when(is_operated & ~missing_arrival_data, num("arr_del15") == 0).alias("is_on_time_arrival"),

        F.when(~is_cancelled, to_int("taxi_out")).alias("taxi_out_min"),
        F.when(is_operated, to_int("taxi_in")).alias("taxi_in_min"),
        to_int("crs_elapsed_time").alias("scheduled_elapsed_min"),
        F.when(is_operated, to_int("actual_elapsed_time")).alias("actual_elapsed_min"),
        F.when(is_operated, to_int("air_time")).alias("air_time_min"),
        to_int("distance").alias("distance_miles"),

        F.when(is_operated, F.coalesce(to_int("carrier_delay"), F.lit(0))).alias("carrier_delay_min"),
        F.when(is_operated, F.coalesce(to_int("weather_delay"), F.lit(0))).alias("weather_delay_min"),
        F.when(is_operated, F.coalesce(to_int("nas_delay"), F.lit(0))).alias("nas_delay_min"),
        F.when(is_operated, F.coalesce(to_int("security_delay"), F.lit(0))).alias("security_delay_min"),
        F.when(is_operated, F.coalesce(to_int("late_aircraft_delay"), F.lit(0))).alias("late_aircraft_delay_min"),

        clean_str("flight_number_reporting_airline").isNull().alias("dq_missing_flight_number"),
        missing_arrival_data.alias("dq_missing_arrival_data"),
        (is_operated & (arr_delay > 1440)).alias("dq_extreme_delay"),
        (is_cancelled & (cancel_code == "D") & (to_int("year") == 2020)).alias("dq_security_code_2020"),
        (is_operated & ~missing_arrival_data).alias("is_delay_metric_eligible"),

        F.col("_source_file").alias("_source_file"),
        F.col("_ingested_at").alias("_bronze_ingested_at"),
        F.current_timestamp().alias("_silver_processed_at"),
    )

    writer = silver.write.format("delta").partitionBy("period_key")

    if silver_exists:
        writer.mode("overwrite").option("partitionOverwriteMode", "dynamic").saveAsTable(SILVER_TABLE)
    else:
        writer.mode("overwrite").saveAsTable(SILVER_TABLE)

    result = spark.table(SILVER_TABLE)
    print(f"Silver table: {SILVER_TABLE}")
    print(f"Columns: {len(result.columns)}")
    print(f"Months in Silver: {result.select('period_key').distinct().count()}")

Months to process: 90
Range: 2019-01 to 2026-06
Silver table: silver_flights
Columns: 54
Months in Silver: 90


In [ ]:
from functools import reduce
from pyspark.sql import functions as F

SILVER_TABLE = "lh_silver.dbo.silver_flights"


def read_bronze(table: str):
    for name in (f"lh_bronze.dbo.{table}", f"lh_bronze.{table}"):
        try:
            return spark.table(name)
        except Exception:
            continue
    raise ValueError(f"Could not read {table} from lh_bronze")


def cnt(condition):
    return F.sum(F.when(condition, 1).otherwise(0))


silver = spark.table(SILVER_TABLE)
bronze = read_bronze("bronze_ontime")
failures = []


def check(name, passed, detail=""):
    print(f"{'PASS' if passed else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    if not passed:
        failures.append(name)


print("=== Strict tests ===")

# 1. Row reconciliation per month
recon = (
    bronze.groupBy(F.col("_period_key").alias("period_key")).agg(F.count("*").alias("bronze_rows"))
    .join(silver.groupBy("period_key").agg(F.count("*").alias("silver_rows")), "period_key", "full_outer")
    .filter(F.coalesce(F.col("bronze_rows"), F.lit(0)) != F.coalesce(F.col("silver_rows"), F.lit(0)))
    .collect()
)
check(
    "1. Row counts match Bronze for every month",
    not recon,
    f"mismatched months: {[(r['period_key'], r['bronze_rows'], r['silver_rows']) for r in recon]}" if recon else "",
)

# 2. Uniqueness
flight_key = ["flight_date", "airline_code", "flight_number", "origin_airport_code", "dest_airport_code", "scheduled_departure_ts"]
duplicate_keys = silver.groupBy(*flight_key).count().filter(F.col("count") > 1).count()
check("2. Each flight appears once", duplicate_keys == 0, f"duplicate keys: {duplicate_keys:,}")

# 3. Status totals against Bronze
b_cancelled = F.col("cancelled").cast("double") == 1
b_diverted = F.col("diverted").cast("double") == 1
bm = bronze.agg(
    F.count("*").alias("rows"),
    cnt(~b_cancelled & ~b_diverted).alias("operated"),
    cnt(b_cancelled).alias("cancelled"),
    cnt(b_diverted).alias("diverted"),
).first().asDict()

never_null = [
    "period_key", "flight_date", "year", "month", "day_of_month", "day_of_week",
    "airline_code", "airline_dot_id", "origin_airport_id", "origin_airport_code",
    "dest_airport_id", "dest_airport_code", "flight_status", "is_cancelled", "is_diverted",
    "scheduled_departure_ts", "scheduled_departure_hour", "scheduled_arrival_hour",
]

delayed = F.col("is_delay_metric_eligible") & F.col("is_arrival_delayed_15")
cause_sum = reduce(
    lambda a, b: a + b,
    [F.col(c) for c in ["carrier_delay_min", "weather_delay_min", "nas_delay_min", "security_delay_min", "late_aircraft_delay_min"]],
)

m = silver.agg(
    F.count("*").alias("rows"),
    cnt(F.col("flight_status") == "Operated").alias("operated"),
    cnt(F.col("flight_status") == "Cancelled").alias("cancelled"),
    cnt(F.col("flight_status") == "Diverted").alias("diverted"),
    *[cnt(F.col(c).isNull()).alias(f"null_{c}") for c in never_null],
    cnt(F.col("is_cancelled") & F.col("cancellation_reason").isNull()).alias("cancelled_without_reason"),
    cnt(~F.col("is_cancelled") & F.col("cancellation_code").isNotNull()).alias("code_on_non_cancelled"),
    cnt(~F.col("is_cancelled") & F.col("actual_departure_ts").isNull()).alias("not_cancelled_without_actual_departure"),
    cnt(
        F.col("is_delay_metric_eligible")
        & ((F.coalesce(F.col("is_on_time_arrival").cast("int"), F.lit(-9))
            + F.coalesce(F.col("is_arrival_delayed_15").cast("int"), F.lit(-9))) != 1)
    ).alias("arrival_flag_conflicts"),
    cnt((F.col("scheduled_departure_hour") < 0) | (F.col("scheduled_departure_hour") > 23)).alias("invalid_departure_hour"),
    cnt(F.col("dq_missing_flight_number")).alias("dq_missing_flight_number"),
    cnt(F.col("dq_missing_arrival_data")).alias("dq_missing_arrival_data"),
    cnt(F.col("dq_extreme_delay")).alias("dq_extreme_delay"),
    cnt(F.col("dq_security_code_2020")).alias("dq_security_code_2020"),
    cnt(F.col("is_delay_metric_eligible")).alias("delay_metric_eligible"),
    cnt(F.to_date("scheduled_departure_ts") != F.col("flight_date")).alias("departure_next_day"),
    cnt(F.col("distance_miles").isNull()).alias("null_distance"),
    cnt(delayed).alias("delayed_15"),
    cnt(delayed & (F.abs(cause_sum - F.col("arrival_delay_nonneg_min")) > 1)).alias("cause_sum_mismatch"),
).first().asDict()

status_keys = ["rows", "operated", "cancelled", "diverted"]
status_issues = {k: (bm[k], m[k]) for k in status_keys if bm[k] != m[k]}
check(
    "3. Status totals match Bronze",
    not status_issues,
    f"(bronze, silver) mismatches: {status_issues}" if status_issues else ", ".join(f"{k} {m[k]:,}" for k in status_keys),
)

# 4. Required fields
null_failures = {c: m[f"null_{c}"] for c in never_null if m[f"null_{c}"] > 0}
check("4. No nulls in required fields", not null_failures, f"nulls found: {null_failures}" if null_failures else "")

# 5. Logical consistency
logic_failures = {
    k: m[k]
    for k in ["cancelled_without_reason", "code_on_non_cancelled", "not_cancelled_without_actual_departure", "arrival_flag_conflicts", "invalid_departure_hour"]
    if m[k] > 0
}
check("5. Logical consistency", not logic_failures, f"violations: {logic_failures}" if logic_failures else "")

print("\n=== Informational checks ===")
print(f"Missing flight number: {m['dq_missing_flight_number']:,}")
print(f"Missing arrival data: {m['dq_missing_arrival_data']:,}")
print(f"Extreme delay over 24h: {m['dq_extreme_delay']:,}")
print(f"2020 security code cancellations: {m['dq_security_code_2020']:,}")
print(f"Delay-metric eligible flights: {m['delay_metric_eligible']:,}")
print(f"Scheduled departures rolled to next day (2400): {m['departure_next_day']:,}")
print(f"Null distance: {m['null_distance']:,}")
print(f"Flights 15+ min late: {m['delayed_15']:,} | causes don't sum to delay: {m['cause_sum_mismatch']:,}")

print(f"\nResult: {'ALL STRICT TESTS PASSED' if not failures else f'{len(failures)} TEST(S) FAILED'}")
if failures:
    raise AssertionError(f"Silver quality gate failed: {failures}")

=== Strict tests ===
PASS | 1. Row counts match Bronze for every month
PASS | 2. Each flight appears once | duplicate keys: 0
PASS | 3. Status totals match Bronze profile | rows 49,251,865, operated 48,068,798, cancelled 1,063,521, diverted 119,546
PASS | 4. No nulls in required fields
PASS | 5. Logical consistency

=== Informational checks ===
Missing flight number: 1 (profile: 1)
Missing arrival data: 5 (profile: 5 missing arr_delay, 5 missing air_time)
Extreme delay over 24h: 3,055 (profile: 3,055)
2020 security code cancellations: 237,189 (profile: 237,189)
Delay-metric eligible flights: 48,068,793
Scheduled departures rolled to next day (2400): 2 (profile: 2)
Null distance: 0
Flights 15+ min late: 9,332,329 | causes don't sum to delay: 97

Result: ALL STRICT TESTS PASSED
